In [ ]:
import os
import sys
import glob
import gc
import numpy as np
import das4whales as dw
import scipy.signal as sp
import matplotlib; matplotlib.use('Agg')
import matplotlib.image as mpimg
from datetime import datetime

try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False
    print("Warning: psutil not installed. Install with: pip install psutil")

In [ ]:
DAS_APL_NAS_SERVER_PATH = "Z:/DAS-APL"
DATA_FOLDER   = "Z:/DAS-APL/data/OOI_RCA2021/Optasence/NorthCable/C3"
OUTPUT_FOLDER = "Z:/DAS-APL/data/OOI_RCA2021_RGBdataset/Optasence/NorthCable/C3"
INTERROGATOR  = 'optasense'

B1 = [20, 28]   # Hz
B2 = [16, 20]   # Hz
B3 = [3, 15]    # Hz

HIGH_PASS    = 3
FILTER_ORDER = 5

T_MIN, T_MAX = None, None   # (s)
D_MIN, D_MAX = None, None   # (m)

CHANNEL_STRIDE = 2

OUTPUT_FORMAT = 'png'
MAX_SIZE      = 1024

In [ ]:
sys.path.append(DAS_APL_NAS_SERVER_PATH)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

h5_files = sorted(glob.glob(os.path.join(DATA_FOLDER, '*.h5')))
print(f'\n{"="*70}')
print(f'DAS CNN IMAGE EXPORT - BATCH PROCESSING')
print(f'{"="*70}')
print(f'Found {len(h5_files)} files')
print(f'Output folder: {OUTPUT_FOLDER}')
print(f'Output format: {OUTPUT_FORMAT}')
if MAX_SIZE:
    print(f'Rescaling: longest axis → {MAX_SIZE} px (aspect ratio preserved)')
else:
    print(f'Rescaling: disabled (original resolution)')
print(f'Python 64-bit: {sys.maxsize > 2**32}')

if HAS_PSUTIL:
    mem = psutil.virtual_memory()
    print(f'Total RAM: {mem.total / 1024**3:.1f} GB')
    print(f'Available RAM: {mem.available / 1024**3:.1f} GB')

print(f'{"="*70}\n')

In [ ]:
def normalize_band(data, perc=90):
    v = np.percentile(np.abs(data), perc)
    return np.clip(np.abs(data) / v, 0, 1)

def crop(data, d_mask, t_mask):
    return data[np.ix_(d_mask, t_mask)]

def log(msg):
    timestamp = datetime.now().strftime("%H:%M:%S")
    if HAS_PSUTIL:
        process = psutil.Process(os.getpid())
        mem_gb = process.memory_info().rss / 1024**3
        print(f'  [{timestamp}] [{mem_gb:.2f}GB] {msg}')
    else:
        print(f'  [{timestamp}] {msg}')

def rescale_image(data, max_size):
    """
    Reescala un array multicanal (H, W, C) de forma que su eje mayor quede en
    max_size píxeles, manteniendo la relación de aspecto.

    Soporta cualquier número de canales C (1, 3, 4, N, ...).
    Para C <= 4 usa PIL directamente; para C > 4 reescala canal a canal.

    data     : np.ndarray shape (H, W, C), valores en [0, 1]
    max_size : int -- tamaño máximo en píxeles del eje mayor
    Returns  : np.ndarray reescalado, shape (H', W', C)
    """
    try:
        from PIL import Image
    except ImportError:
        raise ImportError("Pillow es necesario para reescalar. Instálalo con: pip install Pillow")

    h, w, c = data.shape
    if max(h, w) <= max_size:
        return data

    scale = max_size / max(h, w)
    new_h = max(1, round(h * scale))
    new_w = max(1, round(w * scale))

    img_uint8 = (data * 255).astype(np.uint8)

    if c <= 4:
        rescaled = np.array(
            Image.fromarray(img_uint8).resize((new_w, new_h), Image.LANCZOS)
        ).astype(np.float32) / 255.0
    else:
        rescaled = np.stack([
            np.array(
                Image.fromarray(img_uint8[:, :, i]).resize((new_w, new_h), Image.LANCZOS)
            ).astype(np.float32) / 255.0
            for i in range(c)
        ], axis=-1)

    log(f'Rescaled: ({h}, {w}, {c}) -> ({new_h}, {new_w}, {c})  [scale={scale:.4f}]')
    return rescaled

def save_raw_image(data, output_folder, source_filepath, output_format='png', max_size=None):
    """
    Guarda el array multicanal directamente como imagen sin ejes, bordes ni decoraciones.
    Apto para entrenamiento de CNN.

    data           : np.ndarray de shape (H, W, C), valores en [0, 1]
                     C puede ser cualquier número de bandas (1, 3, 4, N, ...)
    output_folder  : carpeta de destino
    source_filepath: ruta del archivo .h5 origen (se usa para nombrar la imagen)
    output_format  : 'png' -- soportado solo para C <= 4; para C > 4 se fuerza 'npy'
                     'npy' -- guarda el array numpy crudo (siempre válido)
    max_size       : int o None -- si se indica, reescala antes de guardar
    """
    base_name = os.path.splitext(os.path.basename(source_filepath))[0]
    n_bands   = data.shape[2]

    if max_size is not None:
        data = rescale_image(data, max_size)

    if output_format.lower() != 'npy' and n_bands > 4:
        log(f'WARNING: {n_bands} bands detected -- PNG does not support >4 channels. Forcing npy.')
        output_format = 'npy'

    if output_format.lower() == 'npy':
        out_path = os.path.join(output_folder, base_name + '.npy')
        np.save(out_path, (data * 255).astype(np.uint8))
        log(f'Saved numpy array -> {out_path}  shape={data.shape}')
    else:
        out_path = os.path.join(output_folder, base_name + '.png')
        mpimg.imsave(out_path, data)
        log(f'Saved raw PNG -> {out_path}  shape={data.shape}')

    return out_path

In [ ]:
successful = 0
failed     = 0
skipped    = 0

for file_idx, filepath in enumerate(h5_files, 1):

    print(f'\n{"─"*70}')
    print(f'[{datetime.now().strftime("%H:%M:%S")}] [{file_idx}/{len(h5_files)}] {os.path.basename(filepath)}')
    print(f'{"─"*70}')

    try:
        # --- Skip if already processed ---
        base_name    = os.path.splitext(os.path.basename(filepath))[0]
        expected_ext = '.npy' if OUTPUT_FORMAT.lower() == 'npy' else '.png'
        expected_out = os.path.join(OUTPUT_FOLDER, base_name + expected_ext)
        if os.path.exists(expected_out):
            log(f'Already exists, skipping -> {os.path.basename(expected_out)}')
            skipped += 1
            continue

        # --- Read ---
        log('Reading metadata...')
        metadata = dw.data_handle.get_acquisition_parameters(filepath, interrogator=INTERROGATOR)
        fs = metadata['fs']
        dx = metadata['dx']
        nx = metadata['nx']

        selected_channels_m = [0, nx, CHANNEL_STRIDE * dx]
        selected_channels   = [int(v // dx) for v in selected_channels_m]

        log('Loading data...')
        tr, time, dist, fileBeginTimeUTC = dw.data_handle.load_das_data(filepath, selected_channels, metadata)
        log(f'Shape: {tr.shape}  |  fs={fs} Hz  |  dx={dx*CHANNEL_STRIDE} m  |  {len(dist)} channels')

        # --- Filter & bands ---
        log('High-pass filtering...')
        trf = sp.sosfiltfilt(dw.dsp.butterworth_filter([2, HIGH_PASS, 'hp'], fs), tr, axis=-1)

        log('Band filtering B1...')
        tr_b01 = sp.sosfiltfilt(dw.dsp.butterworth_filter([FILTER_ORDER, B1, 'bp'], fs), trf, axis=-1)
        log('Band filtering B2...')
        tr_b02 = sp.sosfiltfilt(dw.dsp.butterworth_filter([FILTER_ORDER, B2, 'bp'], fs), trf, axis=-1)
        log('Band filtering B3...')
        tr_b03 = sp.sosfiltfilt(dw.dsp.butterworth_filter([FILTER_ORDER, B3, 'bp'], fs), trf, axis=-1)

        del trf
        gc.collect()

        # --- Crop ---
        t_mask = np.ones(len(time), dtype=bool) if T_MIN is None else (time >= T_MIN) & (time <= T_MAX)
        d_mask = np.ones(len(dist), dtype=bool) if D_MIN is None else (dist >= D_MIN) & (dist <= D_MAX)

        # --- RGB array ---
        log('Building RGB...')
        r = normalize_band(crop(tr_b01, d_mask, t_mask))
        g = normalize_band(crop(tr_b02, d_mask, t_mask))
        b = normalize_band(crop(tr_b03, d_mask, t_mask))
        rgb = np.stack([r, g, b], axis=-1)   # shape: (n_channels, n_samples, 3)

        del tr, tr_b01, tr_b02, tr_b03, r, g, b
        gc.collect()

        # --- Save raw image ---
        log('Saving raw CNN image...')
        save_raw_image(rgb[::-1], OUTPUT_FOLDER, filepath,
                       output_format=OUTPUT_FORMAT,
                       max_size=MAX_SIZE)

        del rgb
        gc.collect()

        successful += 1

    except OSError as e:
        print(f'  WARNING SKIPPED (corrupted/truncated file): {e}')
        gc.collect()
        skipped += 1
        continue

    except Exception as e:
        print(f'  ERROR: {e}')
        import traceback
        traceback.print_exc()
        gc.collect()
        failed += 1
        continue

    if file_idx % 5 == 0:
        log('Forcing garbage collection...')
        gc.collect()

print(f'\n{"="*70}')
print(f'DONE -- Successful: {successful} | Skipped: {skipped} | Failed: {failed}')
print(f'{"="*70}\n')